# Line following: sharp curves and straight gaps

Experimental extension of the sharp-curve notebook. A blank floor after a recently observed straight, centred line enters bounded forward gap crossing. Tape in both near and middle strips for two successive frames resumes tracking. Losing the line during a turn retains the old bounded search behaviour.

This is open-loop motion during the gap, not measured heading or distance control. It cannot distinguish a genuine gap from missing tape or leaving the track. It does not handle obstacles, green-marker intersections, or finish detection. Use a clear floor with room beyond the gap and a physical stop available. A frozen process/driver can defeat software timeouts.

Preserve your calibrated threshold, band positions, motor speeds, gains, and reverse setting from your working robot notebook. Defaults here are from the Mac copy and may differ. Shut down the previous notebook kernel before opening the camera. Run cells individually; never Run All.

## 1. Initialize once


In [ ]:
import time
import cv2
import numpy as np
import traitlets
import ipywidgets as widgets

from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()

camera = Camera.instance(width=224, height=224)

live_view = widgets.Image(format="jpeg", width=336, height=336)

camera_link = traitlets.dlink(
    (camera, "value"),
    (live_view, "value"),
    transform=bgr8_to_jpeg
)

display(live_view)

In [ ]:
latest = {"sample": None}

def receive_frame(change):
    latest["sample"] = (change["new"], time.monotonic())

camera.observe(receive_frame, names="value")

## 2. Detect three strips
Copy your calibrated threshold below. In preview, verify all strips cover floor, and each green dot follows tape. Farther up the image is farther ahead only for the expected forward/downward camera mounting.


In [ ]:
BLACK_THRESHOLD = 70  # Copy your calibrated threshold here.
# Fractions of image height: change these if the far strip is above the floor.
BANDS = [("near", 0.72, 0.88), ("middle", 0.54, 0.70), ("far", 0.36, 0.52)]

def detect_curve(frame):
    height, width = frame.shape[:2]
    gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, mask = cv2.threshold(gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
    overlay = frame.copy()
    positions = {}
    for name, top, bottom in BANDS:
        y0, y1 = int(top * height), int(bottom * height)
        strip = mask[y0:y1, :]
        contours = cv2.findContours(strip, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
        positions[name] = None
        if contours:
            contour = max(contours, key=cv2.contourArea)
            fraction = cv2.contourArea(contour) / float(strip.size)
            moments = cv2.moments(contour)
            # Permit a broad horizontal segment, but reject an almost dark strip.
            if 0.005 < fraction < 0.80 and moments["m00"] > 0:
                cx = moments["m10"] / moments["m00"]
                cy = int(moments["m01"] / moments["m00"]) + y0
                positions[name] = (cx - width / 2.0) / (width / 2.0)
                cv2.circle(overlay, (int(cx), cy), 4, (0, 255, 0), -1)
        cv2.rectangle(overlay, (0, y0), (width - 1, y1), (255, 0, 0), 1)
        cv2.putText(overlay, name, (3, y0 + 12), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 255), 1)
    cv2.line(overlay, (width // 2, 0), (width // 2, height - 1), (0, 255, 255), 1)
    return positions, overlay, mask


## 3. Steering and bounded turns
Copy your measured minimum reliable speeds. A stopped inner wheel makes a tighter turn. After checking wheel directions, `ALLOW_REVERSE = True` permits an in-place pivot. These parameters require hardware tuning. The direction is latched during a turn and exits only when the near line and steering estimate are centred; a timeout stops unsuccessful turns. A missing line permits at most a short search in the last clear turn direction.


In [ ]:
# Copy your measured, reliable motor settings. These are initial guesses.
BASE_SPEED = 0.15
CURVE_SPEED = 0.13  # Must still reliably move the robot on the floor.
KP = 0.14
MAX_SPEED = 0.25
TURN_SPEED = 0.15
ALLOW_REVERSE = False  # True enables an in-place turn with one wheel reversing.
TURN_ENTER = 0.55
TURN_EXIT = 0.22
TURN_TIMEOUT = 1.0
LOST_TIMEOUT = 0.35
FRAME_TIMEOUT = 0.5

def clamp(value, low, high):
    return max(low, min(high, value))

class CurveController:
    def __init__(self):
        self.turn_direction = 0
        self.turn_started = None
        self.lost_started = None
        self.last_direction = 0
        self.stopped = False

    def stop(self, reason):
        self.stopped = True
        return 0.0, 0.0, "STOP: " + reason

    def pivot(self, direction):
        speed = clamp(TURN_SPEED, 0.0, MAX_SPEED)
        inner = -speed if ALLOW_REVERSE else 0.0
        return (speed, inner) if direction > 0 else (inner, speed)

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        visible = [(name, value) for name, value in positions.items() if value is not None]

        if not visible:
            if self.lost_started is None:
                self.lost_started = now
            if now - self.lost_started >= LOST_TIMEOUT:
                return self.stop("line not reacquired")
            if not self.last_direction:
                return self.stop("line lost without a known turn direction")
            if self.turn_started is None:
                self.turn_started = now
                self.turn_direction = self.last_direction
            if now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")
            left, right = self.pivot(self.turn_direction)
            return left, right, "SEARCH"

        self.lost_started = None
        weights = {"near": 0.45, "middle": 0.35, "far": 0.20}
        error = sum(weights[name] * value for name, value in visible) / sum(weights[name] for name, _ in visible)
        farthest = positions.get("far")
        if farthest is None:
            farthest = positions.get("middle")
        # Anticipate a bend from its displacement relative to the near strip.
        if near is not None and farthest is not None:
            error += 0.40 * (farthest - near)
        error = clamp(error, -1.0, 1.0)
        if abs(error) > 0.15:
            self.last_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            # Require the near line and combined direction to agree before exiting.
            if near is not None and abs(near) < TURN_EXIT and abs(error) < TURN_EXIT:
                self.turn_started = None
                self.turn_direction = 0
            elif now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")

        if self.turn_started is None and abs(error) >= TURN_ENTER:
            self.turn_started = now
            self.turn_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            left, right = self.pivot(self.turn_direction)
            return left, right, "TURN"

        bend = min(abs(error) / TURN_ENTER, 1.0)
        base = BASE_SPEED + (CURVE_SPEED - BASE_SPEED) * bend
        correction = KP * error
        left = clamp(base + correction, 0.0, MAX_SPEED)
        right = clamp(base - correction, 0.0, MAX_SPEED)
        return left, right, "FOLLOW"


## 4. Bounded gap crossing

The project specifies gaps on straight sections, so forward continuation is allowed only after straight-line evidence. Start with a 3–5 cm gap and a generous clear floor around the test. Move up to larger gaps only after short gaps work.

`GAP_SPEED` must reliably move both wheels. `GAP_TIMEOUT` starts when all three strips lose the line and includes time until tape reaches the near and middle strips; camera geometry affects it. Calibrate from measured travel speed and real trials, not the normalized motor command. Keep the timeout only as long as necessary. Do not lower black threshold to make white floor look like tape.


In [ ]:
# Initial trial values; calibrate on the real floor at your working motor power.
GAP_SPEED = BASE_SPEED
GAP_TIMEOUT = 1.0       # Maximum time driving without confirmed near/middle tape.
GAP_STEER_LIMIT = 0.015 # Only a small remembered correction; not heading feedback.
GAP_APPROACH_FRAMES = 3
GAP_APPROACH_MAX_AGE = 0.6
GAP_ALIGNMENT = 0.25   # Normalized offset from image centre.
GAP_STRAIGHT_SPREAD = 0.12
GAP_CONFIRM_FRAMES = 2

class GapCurveController(CurveController):
    def __init__(self):
        super().__init__()
        self.straight_frames = 0
        self.straight_seen = None
        self.saved_correction = 0.0
        self.gap_started = None
        self.confirm_frames = 0

    def gap_commands(self):
        return (clamp(GAP_SPEED + self.saved_correction, 0.0, MAX_SPEED),
                clamp(GAP_SPEED - self.saved_correction, 0.0, MAX_SPEED),
                "GAP: forward, awaiting tape")

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        middle = positions.get("middle")
        values = [value for value in positions.values() if value is not None]

        if self.gap_started is not None:
            # Never reset this deadline because of an isolated noisy detection.
            if now - self.gap_started >= GAP_TIMEOUT:
                return self.stop("gap crossing timed out")
            if near is not None and middle is not None:
                self.confirm_frames += 1
            else:
                self.confirm_frames = 0
            if self.confirm_frames < GAP_CONFIRM_FRAMES:
                return self.gap_commands()
            self.gap_started = None
            self.confirm_frames = 0
            self.lost_started = None
            self.last_direction = 0
            # Resume the existing controller using the newly observed tape.

        if not values:
            recent_straight = (self.straight_seen is not None and
                               now - self.straight_seen <= GAP_APPROACH_MAX_AGE)
            if (self.turn_started is None and recent_straight and
                    self.straight_frames >= GAP_APPROACH_FRAMES):
                self.gap_started = now
                self.straight_frames = 0
                self.straight_seen = None
                self.confirm_frames = 0
                return self.gap_commands()
            return super().step(positions, now)

        left, right, state = super().step(positions, now)
        aligned = max(abs(value) for value in values) <= GAP_ALIGNMENT
        straight = len(values) >= 2 and max(values) - min(values) <= GAP_STRAIGHT_SPREAD
        if state == "FOLLOW" and near is not None and aligned and straight:
            if self.straight_seen is None or now - self.straight_seen > GAP_APPROACH_MAX_AGE:
                self.straight_frames = 0
            self.straight_frames += 1
            self.straight_seen = now
            correction = clamp((left - right) / 2.0, -GAP_STEER_LIMIT, GAP_STEER_LIMIT)
            self.saved_correction = 0.5 * self.saved_correction + 0.5 * correction
        elif state != "FOLLOW" or not aligned or (len(values) >= 2 and not straight):
            self.straight_frames = 0
            self.straight_seen = None
        # A short single-band transition can occur as the line exits the image.
        # Preserve earlier evidence only until GAP_APPROACH_MAX_AGE expires.
        return left, right, state


In [ ]:
overlay_view = widgets.Image(format="jpeg", width=336, height=336)
mask_view = widgets.Image(format="jpeg", width=336, height=336)
status_view = widgets.Label(value="Stationary preview ready")
display(widgets.VBox([widgets.HBox([overlay_view, mask_view]), status_view]))

def run_curves(seconds=10, drive=False):
    robot.stop()
    controller = GapCurveController()
    deadline = time.monotonic() + seconds
    previous_stamp = None
    next_display = 0.0
    try:
        while time.monotonic() < deadline:
            sample = latest["sample"]
            if sample is None:
                print("Stopped: no camera frame. Wait briefly, then retry.")
                break
            frame, captured_at = sample
            now = time.monotonic()
            if now - captured_at > FRAME_TIMEOUT:
                print("Stopped: stale camera frame")
                break
            if captured_at == previous_stamp:
                time.sleep(0.005)
                continue
            previous_stamp = captured_at
            positions, overlay, mask = detect_curve(frame)
            now = time.monotonic()
            if now >= deadline or now - captured_at > FRAME_TIMEOUT:
                print("Stopped: time limit or slow image processing")
                break
            left, right, state = controller.step(positions, now)
            if drive:
                robot.set_motors(left, right)
            if now >= next_display:
                overlay_view.value = bgr8_to_jpeg(overlay)
                mask_view.value = bgr8_to_jpeg(cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR))
                status_view.value = "%s | %s | left %.2f right %.2f" % (
                    "DRIVING" if drive else "PREVIEW", state, left, right)
                next_display = now + 0.1
            if controller.stopped and drive:
                print(state)
                break
            if controller.stopped:
                # Preview remains usable while repositioning the stationary robot.
                controller = GapCurveController()
            time.sleep(0.01)
    finally:
        robot.stop()
    print("Run ended; motors stopped.")


## 4. Stationary preview
Move the robot by hand over the approach and bend. Watch the dots and proposed commands. Right turn: left command exceeds right. Left turn: right exceeds left. No motor commands other than stop are issued in preview.


In [ ]:
run_curves(seconds=20, drive=False)


## 5. Short driving test
After preview works, uncomment the next call to start a two-second test. Start before a broad curve, then test progressively tighter curves in both directions. Keep the robot on the floor with clear space. Change one parameter at a time.


In [ ]:
# Uncomment deliberately to move the robot:
# run_curves(seconds=2, drive=True)


## Gap test and interpretation

1. Copy working calibration, including `ALLOW_REVERSE` if you already enabled it.
2. Run stationary preview. Move along a straight line into a small white gap: expect FOLLOW → GAP → FOLLOW. Preview times out too if movement takes longer than GAP_TIMEOUT.
3. Start an initial two-second drive well before a 3–5 cm gap. Then use `run_curves(seconds=5, drive=True)` when ready; overall run duration must cover approach and crossing.
4. Increase gap length gradually toward the measured requirement (the supplied course document specifies at most 20 cm). Confirm the actual gap length if larger.
5. If GAP stops before reaching tape, measure travel and increase GAP_TIMEOUT in small steps. If it veers sideways, check approach alignment and motor imbalance; a longer timeout will not fix drift. Set GAP_STEER_LIMIT to zero to compare with equal wheel commands.
6. If it enters SEARCH instead of GAP, the recent approach failed the straight/alignment checks. Inspect the three dots; do not loosen the checks blindly.
7. Confirm that missing tape beyond the gap causes a timeout stop, and retest sharp curves in both directions.

Changing parameters alone is sufficient for a new run. If you change BASE_SPEED, rerun `GAP_SPEED = BASE_SPEED` too. No new camera object is needed.


## Tuning

- Line dots wrong: fix lighting, threshold, or band placement before changing steering.
- Gentle bends understeer: increase `KP` slightly; oscillation: reduce it.
- Tight turn enters too late: decrease `TURN_ENTER` gradually, e.g. 0.55 to 0.45.
- Wheel stalls: use the measured reliable motor command; do not keep lowering speed.
- Stopped-wheel turn is too wide: test `ALLOW_REVERSE = True` in a short run after verifying wheel directions.
- Turn timeout: inspect detections and pivot speed before increasing duration. Search is bounded deliberately; do not extend it blindly.
- A true right-angle corner can become invisible or look like a centred horizontal bar. This heuristic is not a complete right-angle/intersection detector. Save failure frames for a dedicated corner state.

## 6. Cleanup
Run when your experiments finish, not between each test. To reopen the camera afterward, restart the kernel and initialize once.


In [ ]:
robot.stop()
camera.unobserve(receive_frame, names="value")
camera_link.unlink()
camera.stop()